# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-29: ECG-Fitness Phase-3 Preprocessing

### What this notebook does
Writes ECG-Fitness into one Phase-3 store in the unified schema (`p3_v1`), directly in
the rechunked uncompressed layout, as an **evaluation-only benchmark**. Part 0 fixes the
benchmark protocol before the store exists. Part 1 builds the store: the face crops
through the app's own capture path, the native ECG on its own clock, and a per-window
reference heart rate computed once and frozen, so no estimator ever influences the label
it is scored against.

No estimator runs in this notebook. The first model number on this corpus comes from the
benchmark script, under the protocol below.

### Decisions carried from NB_P3_28 (all measured there)
- **100 sessions.** 02/04 has no video; 07/02 has a truncated reference and no author
  bounding box, so the authors excluded it and so does this store.
- **Camera c920-2**: larger face (294 against 259 px) and 2.2x steadier per unit face size.
  Phase 2 used c920-1 and had the two cameras the wrong way round.
- **All six activities, the halogen pair 03/04 included.** The Phase-2 exclusion rested on
  an arithmetic error; matched pairs showed halogen brighter in 6 of 6 and no flicker line.
- **Alignment through `c920.csv`**, never proportional resampling: the reference starts
  8.8-1367 s before the video. The frame rate comes from the capture clock in that file,
  not from the container.
- **The reference is a consensus**: a window has one when two of {R-peak intervals,
  QRS-energy autocorrelation, unwrapped device HR} agree within 3 bpm. A single R-peak
  detector read 82 bpm on 09/02 where the truth was 153.
- **The device `ECG HR` column is a signed byte that wraps above 127**, and is unwrapped
  with `% 256` before use.

### Three guards against this store ever being trained on
ECG-Fitness has no PPG waveform, so there is nothing for PhysNet v2's loss to match, and
its licence forbids redistribution, so it must never reach Kaggle. Any one of these keeps
it out of a training run; all three are applied:
1. **It is written to `phase 3 benchmarks\`, not `phase 3 datasets\`.** NB_P3_07's loader
   takes every `.h5` in its folder, and the workstation bench loader every `*_rc_none.h5`.
2. **No `bvp` dataset and no `cardiac_sqi` attribute.** The training loader skips any group
   whose `cardiac_sqi` falls below 0.05 and reads a missing one as 0.0, so leaving it out is
   both honest - there is no BVP to score - and protective. A NaN would *pass* that test.
3. **`role='benchmark'`**, with `has_bvp=False`.

Section 3 checks the first two against the loader's own filter.

## Part 0 - Benchmark protocol (pre-registered 2026-10-03)

Written before the store exists and before any estimator has seen an ECG-Fitness frame,
so nothing in it can be tuned to the result. It is fixed when this notebook is committed;
any later change is dated here and the benchmark is re-run under both versions.

**Question.** When the app cannot see a pulse - violent motion, high heart rate, poor
light - does it decline, or does it report a confident wrong number?

**Data.** The store this notebook writes: 100 sessions, camera c920-2, all six activities.
Nothing is excluded on the estimator side.

**Estimators.** `hr_physnet_v2` (primary) and `hr_spectral` (POS), each through
`estimate(frames, fps)` - the app's own windows, gates and aggregation, at the measured
capture rate. No threshold is changed for this corpus.

**Reference.** The per-window consensus frozen in the store (`ref_hr`, `ref_agree`). A
window has a reference when two readouts agree within 3 bpm; a recording enters the primary
set when at least 80% of its windows have one.

**Primary metrics**, reported per activity and never pooled:
1. *Refusal* - the fraction of recordings with no reported value, and of windows the gates
   discard.
2. *Confident-wrong* - among windows the app keeps, in recordings it reports, the fraction
   off the reference by more than 10 bpm. This is the failure that matters.
3. *Accuracy* on kept windows (MAE, bias), always beside the coverage that produced it.

**Reported separately, never mixed into the primary set:** recordings below the reference
gate; recordings whose crop clamps beyond 2% of its side (the app's framing gate should
refuse them, and whether it does is itself a result); recordings where our detector found
the face in fewer than half of the 12 sampled frames (4 on c920-2; added 2026-10-04, before
any estimator ran), because a box frozen from so few detections makes a refusal partly the
detector's; windows whose real time span is more than 5% off nominal (21 windows, all in
08/05, from the frame clock).

**Stated in advance, so the result can contradict them:**
- H1 - confident-wrong windows concentrate in activities 05 and 06, the highest-motion
  pair: NB_P3_28 found that motion, not light, puts energy into the readout band.
- H2 - refusal rises with motion: 01/03 below 02/04 below 05/06.
- H3 - the halogen sessions do no worse than their ambient partners, 03 against 01 and
  04 against 02.

**What it cannot show:** a high-HR bias separated from motion, because the two arrive
together in this corpus; HRV or respiration, for which there is no reference; anything
clinical.

## Imports & Setup

In [1]:
import re
import sys
import time
from pathlib import Path

import cv2
import h5py
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

# Walk up from the notebook until the repo root is found, identified by the MediaPipe
# asset (the same test NB_P3_25, NB_P3_27 and NB_P3_28 use).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The crop is the app's own capture path, imported rather than copied: the store crops
# exactly as the app crops an uploaded video, so train/serve parity is structural.
from app.live_vitals.config import (CLIP_LEN, MAX_BOX_SIDE_LOST, N_DETECT_FRAMES,
                                    TARGET_SIZE, WINDOW_STRIDE)
from app.live_vitals.capture.session import crops_from_video, rate_verdict, stable_box
from app.live_vitals.preprocess.face_box import box_clamp_report, make_landmarker

ECGF_ROOT = Path(r'F:\ECG_Fitness_dataset\ecg-fitness_raw-v1.0')
OUT_DIR = Path(r'D:\QualityPhys\phase 3 benchmarks')   # NOT 'phase 3 datasets' - see guards
DATASET_NAME = 'ecg_fitness'
OUT_H5 = OUT_DIR / f'{DATASET_NAME}_phase3_rc_none.h5'
LOG_PATH = REPO_ROOT / 'Data' / f'processing_log_{DATASET_NAME}_phase3.csv'

CAMERA = 'c920-2'              # NB_P3_28 section 5
ROLE = 'benchmark'             # evaluation only; never train
SCHEMA_VERSION = 'p3_v1'
FRAMES_PER_CHUNK = 32          # the layout scripts/rechunk_h5.py produces
EXPECTED_FRAMES = 1800         # 60 s at ~30 fps; any other count is treated as corruption
EXCLUDED = {('07', '02'): 'truncated reference; no author bounding box (excluded by the authors)'}

ECG_SCALE = 65536.0            # the Viatom ECG column is fixed point (NB_P3_28 section 2)
REF_FS_RANGE = (100.0, 150.0)  # a Viatom record outside this is not a 125 Hz record
HR_MIN_BPM, HR_MAX_BPM = 35.0, 220.0
QRS_LOW_HZ, QRS_HIGH_HZ = 5.0, 20.0
HR_AGREE_BPM = 3.0             # two readouts within this make a window reference
REF_RECORDING_MIN = 0.80       # a recording needs this fraction of windows referenced
SPAN_TOLERANCE = 0.05          # windows whose real span is further off nominal are flagged
FRAME_DARK, FRAME_BRIGHT = 5.0, 250.0  # exposure gate, as NB_P3_25 and NB_P3_27

# Lighting and motion class per activity id, both measured in NB_P3_28: 03 and 04 are
# 1.26-2.47x brighter than their partners, and head motion pairs the ids as 01/03 (low),
# 02/04 (moderate) and 05, 06 (high). The activity names are still unestablished.
LIGHTING = {'01': 'ambient', '02': 'ambient', '03': 'halogen',
            '04': 'halogen', '05': 'ambient', '06': 'ambient'}
ACTIVITY_PAIR = {'01': '01/03', '03': '01/03', '02': '02/04', '04': '02/04',
                 '05': '05', '06': '06'}
MOTION_CLASS = {'01': 'low', '03': 'low', '02': 'moderate', '04': 'moderate',
                '05': 'high', '06': 'high'}

OVERWRITE = False
MAX_RECORDS = None             # set to an int for a trial run

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('ECG-Fitness root:', ECGF_ROOT, '| exists:', ECGF_ROOT.exists())
print('Output store    :', OUT_H5, '(exists)' if OUT_H5.exists() else '(new)')
print('Log             :', LOG_PATH)
print(f'camera {CAMERA} | role {ROLE} | clip {CLIP_LEN} frames, stride {WINDOW_STRIDE}')

ECG-Fitness root: F:\ECG_Fitness_dataset\ecg-fitness_raw-v1.0 | exists: True
Output store    : D:\QualityPhys\phase 3 benchmarks\ecg_fitness_phase3_rc_none.h5 (exists)
Log             : D:\code\QualityPhys - CRVSE Project\Data\processing_log_ecg_fitness_phase3.csv
camera c920-2 | role benchmark | clip 160 frames, stride 80


## 1. Reading the reference (identical to NB_P3_28)
The functions below are NB_P3_28's, unchanged, so the frozen reference reproduces the
numbers that notebook reported: 1883 of 2100 windows referenced, 83 of 100 recordings at
80% or more. Section 3 checks that it does.

`reference_arrays` computes the reference once per recording on the app's window grid -
`CLIP_LEN` frames at `WINDOW_STRIDE`, the grid `check_ubfc_regression.window_starts`
produces - and keys every value by its start frame. It also records each window's real
time span from the capture clock, because the estimator will assume `CLIP_LEN / fps` and a
window that really ran 6% faster reads its heart rate 6% low.

In [2]:
def read_viatom(path):
    """Reads viatom-raw.csv, dropping any row that did not parse completely.

    One recording (07/02) ends mid-line, and pandas turns that fragment into a row
    carrying a timestamp and nothing else. The count of dropped rows is kept on the
    frame so the caller can refuse the file rather than silently repair it.
    """
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    complete = df.notna().all(axis=1)
    out = df.loc[complete].astype('int64').reset_index(drop=True)
    out.attrs['dropped_rows'] = int((~complete).sum())
    return out


def read_sync(path):
    """Reads c920.csv as (frame wall-clock ms, Viatom row index), one row per frame."""
    a = np.loadtxt(path, delimiter=',', dtype=np.int64)
    return a[:, 0], a[:, 1]


def device_hr(df):
    """Decodes the Viatom 'ECG HR' column, a signed byte that wraps above 127.

    135 bpm is stored as -121. After unwrapping, values outside the physiological band
    (0 while the device is still computing, 255 from its -1 sentinel) become NaN.
    """
    hr = df['ECG HR'].to_numpy() % 256
    return np.where((hr >= HR_MIN_BPM) & (hr <= HR_MAX_BPM), hr.astype(float), np.nan)


def grid_deviation_ms(t_ms):
    """Returns (rms, max absolute) deviation of frame times from a uniform grid, in ms."""
    grid = np.linspace(t_ms[0], t_ms[-1], len(t_ms))
    d = t_ms - grid
    return float(np.sqrt(np.mean(d ** 2))), float(np.max(np.abs(d)))


def window_starts(n_frames):
    """The app's analysis-window starts (check_ubfc_regression.window_starts)."""
    return range(0, n_frames - CLIP_LEN + 1, WINDOW_STRIDE)


def qrs_energy(ecg, fs):
    """Band-passes the ECG to the QRS band, squares it and integrates over 120 ms.

    Squaring makes the result independent of lead polarity, which the dataset does not
    document and which is not constant across subjects.
    """
    x = np.asarray(ecg, dtype=np.float64)
    b, a = butter(2, [QRS_LOW_HZ / (fs / 2), QRS_HIGH_HZ / (fs / 2)], btype='band')
    win = max(1, int(0.12 * fs))
    return np.convolve(filtfilt(b, a, x - x.mean()) ** 2, np.ones(win) / win, mode='same')


def rate_from_peaks(energy, fs):
    """Heart rate from the median interval between thresholded QRS-energy peaks."""
    if energy.size < int(2 * fs):
        return np.nan, np.nan
    threshold = np.median(energy) + 0.5 * (np.percentile(energy, 98) - np.median(energy))
    peaks, _ = find_peaks(energy, height=threshold,
                          distance=max(1, int(60.0 / HR_MAX_BPM * fs)))
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)]
    if ibi.size < 3:
        return np.nan, np.nan
    med = float(np.median(ibi))
    return 60.0 / med, float(np.median(np.abs(ibi - med)) / med)


def rate_from_autocorr(energy, fs):
    """Heart rate from the dominant period of the QRS energy, with its peak height.

    The autocorrelation is divided by the overlap count at each lag, because the raw sum
    shrinks with lag and would bias every window towards a short period.
    """
    x = np.asarray(energy, dtype=np.float64) - np.mean(energy)
    if x.size < 32 or x.std() < 1e-12:
        return np.nan, 0.0
    ac = np.correlate(x, x, 'full')[x.size - 1:] / np.arange(x.size, 0, -1)
    if ac[0] <= 0:
        return np.nan, 0.0
    ac = ac / ac[0]
    lo, hi = int(60.0 / HR_MAX_BPM * fs), min(int(60.0 / HR_MIN_BPM * fs), len(ac) - 2)
    if hi <= lo + 1:
        return np.nan, 0.0
    k = lo + int(np.argmax(ac[lo:hi + 1]))
    height = float(ac[k])
    if 0 < k < len(ac) - 1:
        curve = ac[k - 1] - 2 * ac[k] + ac[k + 1]
        if abs(curve) > 1e-12:
            k = k + 0.5 * (ac[k - 1] - ac[k + 1]) / curve
    return float(60.0 * fs / k), height


def consensus_hr(values, tol=HR_AGREE_BPM):
    """Returns (HR, how many readouts agreed) for the largest cluster within `tol`."""
    finite = [v for v in values if np.isfinite(v)]
    best = []
    for i in range(len(finite)):
        group = [v for v in finite if abs(v - finite[i]) <= tol]
        if len(group) > len(best):
            best = group
    return (float(np.median(best)), len(best)) if len(best) >= 2 else (np.nan, len(best))


def reference_arrays(v, idx, t_frame, fs):
    """The frozen per-window reference, as arrays keyed by window start frame."""
    energy = qrs_energy(v['ECG'].to_numpy() / ECG_SCALE, fs)
    dev = device_hr(v)
    rows = []
    for k in window_starts(len(t_frame)):
        lo = int(np.clip(idx[k], 0, len(v) - 1))
        hi = int(np.clip(idx[k + CLIP_LEN - 1], 0, len(v) - 1))
        seg = energy[lo:hi + 1]
        hr_peaks, disp = rate_from_peaks(seg, fs)
        hr_auto, periodicity = rate_from_autocorr(seg, fs)
        window_dev = dev[lo:hi + 1]
        hr_device = float(np.nanmedian(window_dev)) if np.isfinite(window_dev).any() else np.nan
        hr, agree = consensus_hr([hr_peaks, hr_auto, hr_device])
        rows.append((k, hr, agree, hr_peaks, hr_auto, hr_device, disp, periodicity,
                     bool(0 <= idx[k] and idx[k + CLIP_LEN - 1] < len(v)),
                     (t_frame[k + CLIP_LEN - 1] - t_frame[k]) / 1000.0))
    cols = list(zip(*rows))
    return dict(
        ref_window_start=np.asarray(cols[0], dtype=np.int32),
        ref_hr=np.asarray(cols[1], dtype=np.float32),
        ref_agree=np.asarray(cols[2], dtype=np.int8),
        ref_hr_peaks=np.asarray(cols[3], dtype=np.float32),
        ref_hr_auto=np.asarray(cols[4], dtype=np.float32),
        ref_hr_device=np.asarray(cols[5], dtype=np.float32),
        ref_rr_disp=np.asarray(cols[6], dtype=np.float32),
        ref_periodicity=np.asarray(cols[7], dtype=np.float32),
        ref_covered=np.asarray(cols[8], dtype=bool),
        ref_span_s=np.asarray(cols[9], dtype=np.float32))


sessions = sorted((p.parent.name, p.name)
                  for p in ECGF_ROOT.glob('[0-9][0-9]/[0-9][0-9]')
                  if (p / f'{CAMERA}.avi').exists())
queued = [s for s in sessions if s not in EXCLUDED]
print(f'sessions with a {CAMERA} video: {len(sessions)} | excluded: {sorted(EXCLUDED)} | '
      f'queued: {len(queued)}')

sessions with a c920-2 video: 101 | excluded: [('07', '02')] | queued: 100


## 2. What the container claims
Reads the declared frame rate and frame count of every `c920-2` video from its header
alone - no frame is decoded - and sets them beside the capture clock in `c920.csv`. The app
decimates by the container's declared rate before it measures anything, so a header that
declared 60 fps would silently halve the store; `build_record` raises on any frame count
other than 1800, and this is where such a header would show first.

In [3]:
rows = []
for subject, activity in queued:
    folder = ECGF_ROOT / subject / activity
    cap = cv2.VideoCapture(str(folder / f'{CAMERA}.avi'))
    declared = cap.get(cv2.CAP_PROP_FPS)
    n_header = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    t_frame, _ = read_sync(folder / 'c920.csv')
    measured = (len(t_frame) - 1) / ((t_frame[-1] - t_frame[0]) / 1000)
    rows.append(dict(subject=subject, activity=activity, declared_fps=declared,
                     measured_fps=measured, header_frames=n_header, sync_rows=len(t_frame),
                     width=width, height=height))
container = pd.DataFrame(rows)

print(f'declared fps: {sorted(container.declared_fps.round(3).unique().tolist())}')
print(f'measured fps: {container.measured_fps.min():.3f}-{container.measured_fps.max():.3f}')
print(f'header frames: {sorted(container.header_frames.unique().tolist())} | '
      f'sync rows: {sorted(container.sync_rows.unique().tolist())}')
print(f'frame size: {sorted(set(zip(container.width, container.height)))}')
gap = (container.declared_fps - container.measured_fps).abs() / container.measured_fps
print(f'container rate off the capture clock by more than 2%: {int((gap > 0.02).sum())} videos'
      f' - the store uses the capture clock')
mismatch = container[(container.header_frames != EXPECTED_FRAMES) | (container.sync_rows != EXPECTED_FRAMES)]
print('frame-count mismatches:', mismatch[['subject', 'activity', 'header_frames', 'sync_rows']]
      .to_dict('records') if len(mismatch) else 'none')

declared fps: [30.0]
measured fps: 28.158-30.016
header frames: [1800] | sync rows: [1800]
frame size: [(1920, 1080)]
container rate off the capture clock by more than 2%: 2 videos - the store uses the capture clock
frame-count mismatches: none


In [4]:
# Does the app's own detector see the face on both cameras? NB_P3_28 chose c920-2 on the
# authors' boxes, which found a face in every frame of both cameras. The store crops with
# MediaPipe instead, and in the trial run it found the face in only 3 of the 12 sampled
# frames of 00/02. A refusal caused by the detector is not the estimator's, so the store
# should use the camera our detector sees best. Decision rule, stated before the numbers:
# keep c920-2 unless c920-1 finds the face in clearly more sampled frames on the moving
# activities (02, 04, 05, 06).
from app.live_vitals.preprocess.face_box import landmark_box

LANDMARKER = globals().get('LANDMARKER') or make_landmarker()


def sampled_detections(video):
    """Counts MediaPipe detections on the 12 frames crops_from_video samples."""
    cap = cv2.VideoCapture(str(video))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_at = sorted(set(np.linspace(0, n_total - 1, N_DETECT_FRAMES).astype(int)))
    found = 0
    for i in sample_at:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, bgr = cap.read()
        if ok and landmark_box(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), LANDMARKER) is not None:
            found += 1
    cap.release()
    return found


t0 = time.perf_counter()
det = pd.DataFrame([dict(subject=s, activity=a, camera=cam,
                         detected=sampled_detections(ECGF_ROOT / s / a / f'{cam}.avi'))
                    for s, a in queued for cam in ('c920-1', 'c920-2')])
print(f'{len(det)} videos, {N_DETECT_FRAMES} sampled frames each, '
      f'{(time.perf_counter() - t0) / 60:.1f} min')
print(det.pivot_table(index='activity', columns='camera', values='detected',
                      aggfunc=['median', 'min']).to_string())
print('\nvideos with the face in fewer than half the samples:',
      det[det.detected < N_DETECT_FRAMES / 2].groupby('camera').size().to_dict() or 'none')
wide = det.pivot_table(index=['subject', 'activity'], columns='camera', values='detected')
moving = wide[wide.index.get_level_values('activity').isin(['02', '04', '05', '06'])]
print(f'moving activities: c920-2 sees more in {int((moving["c920-2"] > moving["c920-1"]).sum())}, '
      f'c920-1 in {int((moving["c920-1"] > moving["c920-2"]).sum())}, '
      f'tied in {int((moving["c920-1"] == moving["c920-2"]).sum())} of {len(moving)}')

200 videos, 12 sampled frames each, 5.1 min
         median           min       
camera   c920-1 c920-2 c920-1 c920-2
activity                            
01         11.0   12.0      0     11
02          8.0    9.5      2      3
03         11.0   12.0      0      7
04          7.0    9.0      3      5
05         12.0   12.0     11     11
06         12.0   12.0      9      3

videos with the face in fewer than half the samples: {'c920-1': 16, 'c920-2': 4}
moving activities: c920-2 sees more in 30, c920-1 in 12, tied in 24 of 66


**Camera check: c920-2 confirmed (2026-10-04).** On the 12 frames `crops_from_video` samples,
our detector finds the face at least as often on c920-2 as on c920-1 in every activity by
median - 9.5 against 8 on 02 and 9 against 7 on 04, the two where it struggles most - and in
30 of the 66 moving-activity recordings it finds it more often, against 12 the other way.
Four c920-2 videos keep the face in fewer than half the samples, against 16 on c920-1, and
c920-1 has recordings in activities 01 and 03 where it finds none at all, which
`crops_from_video` would refuse outright. The one place c920-1 does better is the worst 06
recording (9 detections against 3). The decision rule stated above holds and the store stays
on c920-2. Rowing (02/04, on NB_P3_28's reading) is where detection is weakest on either
camera, so section 5 lists the low-detection recordings and the protocol reports them
separately.

## 3. Store schema and the single-recording walkthrough

### Schema (ECG-Fitness)
| name | source | shape, type | notes |
|------|--------|-------------|-------|
| `frames` | c920-2 AVI, face crop | [1800, 72, 72, 3] uint8 | the app's `crops_from_video`, frames in capture order |
| `frame_t_ms` | `c920.csv` | [1800] int64 | capture clock, ms after the first Viatom sample |
| `frame_ecg_idx` | `c920.csv` | [1800] int32 | the Viatom row each frame maps to |
| `ecg` | `viatom-raw.csv` | [N] float32 | native ~125 Hz, fixed point divided by 2^16 |
| `ecg_ms` | `viatom-raw.csv` | [N] int64 | the Viatom's own clock, same origin as `frame_t_ms` |
| `ecg_hr_device` | `viatom-raw.csv` | [N] float32 | unwrapped device HR; NaN outside 35-220 |
| `ref_*` | computed once here | [21] each | the frozen per-window reference, keyed by `ref_window_start` |

The `ref_` datasets: `ref_hr` and `ref_agree` (the consensus and how many readouts agreed),
the three readouts `ref_hr_peaks`, `ref_hr_auto`, `ref_hr_device`, `ref_rr_disp`,
`ref_periodicity`, `ref_covered`, and `ref_span_s`, the window's real duration.

Key attributes: `role='benchmark'`, `camera`, `activity`, `activity_pair`, `lighting`,
`motion_class`; `fps` from the capture clock beside `fps_container`; `grid_rms_ms`;
`ref_accepted_frac` and `ref_ok`; the crop box, `box_side_lost` (the store convention,
worse axis) beside `app_frac_side_lost` (what the app's framing gate reads); and
`capture_verdict`, the app's own ACCEPT / WARN / REJECT for the file. There is **no
`cardiac_sqi`** and **no `bvp`** - see the guards above.

`usable` gates on exposure only, as in every Phase-3 store. A recording the app's framing
gate would refuse is still written, flagged, because whether the app refuses it is part of
what the benchmark measures.

### The box
`crops_from_video` does not return its box, and every Phase-3 store keeps one. `sampled_box`
samples the same 12 frames and calls the same `stable_box`, and `build_record` checks that
the recomputed box clamps exactly as the one the crop used before anything is written.

In [5]:
LANDMARKER = make_landmarker()


def sampled_box(video):
    """Recomputes the box crops_from_video used, from the same 12 sampled frames."""
    cap = cv2.VideoCapture(str(video))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total <= 0:
        cap.release()
        raise RuntimeError('container reports no frame count')
    sample_at = sorted(set(np.linspace(0, n_total - 1, N_DETECT_FRAMES).astype(int)))
    samples = []
    for i in sample_at:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, bgr = cap.read()
        if ok:
            samples.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    box, detections = stable_box(samples, LANDMARKER)
    return box, detections, len(sample_at)


def box_side_lost(box):
    """Fraction of the box side outside the frame, on the worse axis (as NB_P3_27)."""
    lost_x = max(0.0, -box[0]) + max(0.0, box[2] - 1.0)
    lost_y = max(0.0, -box[1]) + max(0.0, box[3] - 1.0)
    return float(max(lost_x / (box[2] - box[0]), lost_y / (box[3] - box[1])))


def build_record(subject, activity):
    """Builds one unified-schema record for an ECG-Fitness session."""
    t_start = time.perf_counter()
    folder = ECGF_ROOT / subject / activity
    video = folder / f'{CAMERA}.avi'
    t_frame, idx = read_sync(folder / 'c920.csv')
    v = read_viatom(folder / 'viatom-raw.csv')
    if v.attrs['dropped_rows']:
        raise RuntimeError(f"{v.attrs['dropped_rows']} unparsable reference row(s)")
    t_ref = v['milliseconds'].to_numpy()
    ecg_fs = (len(v) - 1) / ((t_ref[-1] - t_ref[0]) / 1000)
    if not REF_FS_RANGE[0] <= ecg_fs <= REF_FS_RANGE[1]:
        raise RuntimeError(f'reference rate {ecg_fs:.3g} Hz')

    # The app's own capture path. Its rate comes from container timestamps, which an
    # uncompressed AVI only synthesises from the declared rate; the store's fps is taken
    # from the capture clock below instead.
    frames, fps_stamps, quality = crops_from_video(video, LANDMARKER)
    if len(frames) != len(t_frame) or len(frames) != EXPECTED_FRAMES:
        raise RuntimeError(f'{len(frames)} frames decoded against {len(t_frame)} sync rows')
    box, detections, attempted = sampled_box(video)
    report = box_clamp_report(box, quality.width, quality.height)
    if box is None or not np.isclose(report['frac_side_lost'], quality.frac_side_lost):
        raise RuntimeError('recomputed box differs from the one crops_from_video used')

    fps = (len(t_frame) - 1) / ((t_frame[-1] - t_frame[0]) / 1000)
    dt = np.diff(t_frame)
    rms_ms, max_ms = grid_deviation_ms(t_frame.astype(float))
    ref = reference_arrays(v, idx, t_frame, ecg_fs)
    accepted = ref['ref_agree'] >= 2
    nominal_span = (CLIP_LEN - 1) / fps
    off_span = np.abs(ref['ref_span_s'] / nominal_span - 1.0) > SPAN_TOLERANCE
    brightness = float(np.median([frames[i].mean()
                                  for i in np.linspace(0, len(frames) - 1, 9).astype(int)]))
    frame_ok = bool(FRAME_DARK <= brightness <= FRAME_BRIGHT)
    rate_true, _ = rate_verdict(fps)

    attrs = dict(
        schema_version=SCHEMA_VERSION, dataset='ECG-Fitness', role=ROLE, split=ROLE,
        camera=CAMERA, source_file=f'{subject}/{activity}/{CAMERA}.avi',
        subject_id=subject, subject_uid=f'ecg_fitness_{subject}', activity=activity,
        activity_pair=ACTIVITY_PAIR[activity], lighting=LIGHTING[activity],
        motion_class=MOTION_CLASS[activity],
        fps=float(fps), fps_container=float(quality.source_fps), fps_stamps=float(fps_stamps),
        n_frames=int(len(frames)), duration_s=float((t_frame[-1] - t_frame[0]) / 1000),
        frame_size=int(TARGET_SIZE), frame_width=int(quality.width),
        frame_height=int(quality.height),
        grid_rms_ms=rms_ms, grid_max_ms=max_ms, frame_dt_max_ms=float(dt.max()),
        frames_dt_gt_50ms=int((dt > 50).sum()),
        has_frames=True, has_bvp=False, has_ecg=True, has_hr=True,
        has_resp=False, has_spo2=False, has_eda=False,
        time_origin='first Viatom sample',
        ecg_fs=float(ecg_fs), ecg_rows=int(len(v)),
        ecg_offset_s=float((t_frame[0] - t_ref[0]) / 1000),
        ecg_coverage=float(((idx >= 0) & (idx < len(v))).mean()),
        ref_rule=f'two of R-peak intervals, QRS-energy autocorrelation and device HR '
                 f'within {HR_AGREE_BPM:g} bpm',
        ref_windows=int(len(accepted)), ref_accepted=int(accepted.sum()),
        ref_accepted_frac=float(accepted.mean()),
        ref_ok=bool(accepted.mean() >= REF_RECORDING_MIN),
        ref_hr_median=float(np.nanmedian(ref['ref_hr'])) if accepted.any() else float('nan'),
        ref_hr_max=float(np.nanmax(ref['ref_hr'])) if accepted.any() else float('nan'),
        ref_windows_off_span=int(off_span.sum()),
        box_rx0=float(box[0]), box_ry0=float(box[1]), box_rx1=float(box[2]), box_ry1=float(box[3]),
        box_side_lost=box_side_lost(box), app_frac_side_lost=float(quality.frac_side_lost),
        crop_aspect=float(quality.crop_aspect),
        face_detections=int(detections), face_detections_attempted=int(attempted),
        capture_verdict=str(quality.verdict), capture_notes='; '.join(quality.notes),
        rate_verdict_true=str(rate_true),
        brightness=brightness, frame_ok=frame_ok, usable=frame_ok,
        seconds_to_build=round(time.perf_counter() - t_start, 1))
    arrays = dict(
        frames=frames,
        frame_t_ms=(t_frame - t_ref[0]).astype(np.int64),
        frame_ecg_idx=idx.astype(np.int32),
        ecg=(v['ECG'].to_numpy() / ECG_SCALE).astype(np.float32),
        ecg_ms=(t_ref - t_ref[0]).astype(np.int64),
        ecg_hr_device=device_hr(v).astype(np.float32),
        **ref)
    return f'{subject}_{activity}', arrays, attrs


demo_name, demo_arrays, demo_attrs = build_record(*queued[0])
print('record:', demo_name)
for k, val in demo_attrs.items():
    print(f'  {k:26s} {round(val, 4) if isinstance(val, float) else val}')
print({k: (a.shape, str(a.dtype)) for k, a in demo_arrays.items()})

record: 00_01
  schema_version             p3_v1
  dataset                    ECG-Fitness
  role                       benchmark
  split                      benchmark
  camera                     c920-2
  source_file                00/01/c920-2.avi
  subject_id                 00
  subject_uid                ecg_fitness_00
  activity                   01
  activity_pair              01/03
  lighting                   ambient
  motion_class               low
  fps                        30.0093
  fps_container              30.0
  fps_stamps                 30.0
  n_frames                   1800
  duration_s                 59.948
  frame_size                 72
  frame_width                1920
  frame_height               1080
  grid_rms_ms                11.3209
  grid_max_ms                26.7122
  frame_dt_max_ms            70.0
  frames_dt_gt_50ms          75
  has_frames                 True
  has_bvp                    False
  has_ecg                    True
  has_hr           

## 4. Write function and batch processing
Each recording becomes a group named `<subject>_<activity>` in one consolidated HDF5:
`frames` in `(32, 72, 72, 3)` chunks, uncompressed, every other dataset unchunked. Append
mode with skip-existing makes the run resumable, and each group is flushed as soon as it is
written. Each session reads about 5.6 GB of uncompressed video from the T7, so the full
batch takes on the order of an hour; `MAX_RECORDS` limits a trial run.

The log goes to `Data/processing_log_ecg_fitness_phase3.csv`. It carries recording-level
numbers only - no frames and no signals - so it falls outside the licence's redistribution
limit in the same way the Phase-2 ECG-Fitness logs do.

In [6]:
def write_group(store, name, arrays, attrs):
    """Writes one recording as a group: frames chunked along time, then all attributes."""
    g = store.create_group(name)
    for key, value in arrays.items():
        if key == 'frames':
            g.create_dataset(key, data=value,
                             chunks=(min(FRAMES_PER_CHUNK, len(value)),) + value.shape[1:])
        else:
            g.create_dataset(key, data=value)
    for key, value in attrs.items():
        g.attrs[key] = value


LOG_COLS = ('subject_id', 'activity', 'lighting', 'motion_class', 'n_frames', 'fps',
            'fps_container', 'grid_rms_ms', 'usable', 'brightness', 'face_detections',
            'box_side_lost', 'app_frac_side_lost', 'capture_verdict', 'ref_accepted',
            'ref_windows', 'ref_accepted_frac', 'ref_ok', 'ref_hr_median', 'ref_hr_max',
            'ref_windows_off_span', 'seconds_to_build', 'status', 'error')


def process_all(sessions_to_build, out_h5, limit=None, overwrite=False):
    """Writes every queued session into one consolidated HDF5, one group each."""
    todo = sessions_to_build[:limit] if limit else sessions_to_build
    rows = []
    t0 = time.perf_counter()
    with h5py.File(out_h5, 'a') as store:
        for i, (subject, activity) in enumerate(todo, 1):
            name = f'{subject}_{activity}'
            if name in store and not overwrite:
                a = dict(store[name].attrs)
                a.update(status='skipped', error='')
            else:
                try:
                    _, arrays, attrs = build_record(subject, activity)
                    if name in store:
                        del store[name]
                    write_group(store, name, arrays, attrs)
                    store.flush()
                    a = dict(attrs, status='ok', error='')
                except Exception as exc:
                    a = dict(subject_id=subject, activity=activity, usable=False,
                             status='error', error=str(exc))
            print(f'   {i}/{len(todo)} {name} -> {a["status"]}'
                  f'{" (" + a["error"] + ")" if a["error"] else ""} | '
                  f'{(time.perf_counter() - t0) / 60:.1f} min', flush=True)
            rows.append({c: a.get(c) for c in LOG_COLS})
    return pd.DataFrame(rows)


log = process_all(queued, OUT_H5, limit=MAX_RECORDS, overwrite=OVERWRITE)
log.to_csv(LOG_PATH, index=False)
print('Store:', OUT_H5, f'({OUT_H5.stat().st_size / 1e9:.2f} GB)')
print('ok:', int((log.status == 'ok').sum()), '| skipped:', int((log.status == 'skipped').sum()),
      '| errors:', int((log.status == 'error').sum()))
print('Log ->', LOG_PATH)

   1/100 00_01 -> skipped | 0.0 min
   2/100 00_02 -> skipped | 0.0 min
   3/100 00_03 -> skipped | 0.0 min
   4/100 00_04 -> skipped | 0.0 min
   5/100 00_05 -> skipped | 0.0 min
   6/100 00_06 -> skipped | 0.0 min
   7/100 01_01 -> skipped | 0.0 min
   8/100 01_02 -> skipped | 0.0 min
   9/100 01_03 -> skipped | 0.0 min
   10/100 01_04 -> skipped | 0.0 min
   11/100 01_05 -> skipped | 0.0 min
   12/100 01_06 -> skipped | 0.0 min
   13/100 02_01 -> skipped | 0.0 min
   14/100 02_02 -> skipped | 0.0 min
   15/100 02_03 -> skipped | 0.0 min
   16/100 02_05 -> skipped | 0.0 min
   17/100 02_06 -> skipped | 0.0 min
   18/100 03_01 -> skipped | 0.0 min
   19/100 03_02 -> skipped | 0.0 min
   20/100 03_03 -> skipped | 0.0 min
   21/100 03_04 -> skipped | 0.0 min
   22/100 03_05 -> skipped | 0.0 min
   23/100 03_06 -> skipped | 0.0 min
   24/100 04_01 -> skipped | 0.0 min
   25/100 04_02 -> skipped | 0.0 min
   26/100 04_03 -> skipped | 0.0 min
   27/100 04_04 -> skipped | 0.0 min
   28/100 

## 5. Output validation
Re-opens the store and checks every group against the schema, then three things that make
the store trustworthy as a benchmark rather than merely complete:

- **The guards hold.** NB_P3_07's own index filter is applied to the store; it must admit
  nothing.
- **The reference reproduces NB_P3_28.** With all 100 recordings present, the frozen
  reference must give exactly 1883 referenced windows and 83 recordings at 80% or more.
  A difference means the reference code or the input changed, and nothing downstream
  should run until it is explained.
- **What the app's capture gate makes of each file**, recorded before any estimator runs,
  because the protocol reports clamped recordings separately.

These are checks on the stored data, not an evaluation of any method.

In [7]:
NB_P3_28_ACCEPTED, NB_P3_28_RECORDINGS_OK = 1883, 83   # NB_P3_28 section 4
REF_KEYS = {'ref_window_start', 'ref_hr', 'ref_agree', 'ref_hr_peaks', 'ref_hr_auto',
            'ref_hr_device', 'ref_rr_disp', 'ref_periodicity', 'ref_covered', 'ref_span_s'}
EXPECTED_SETS = {'frames', 'frame_t_ms', 'frame_ecg_idx', 'ecg', 'ecg_ms', 'ecg_hr_device'} | REF_KEYS
N_WINDOWS = len(window_starts(EXPECTED_FRAMES))

with h5py.File(OUT_H5, 'r') as store:
    groups = sorted(store.keys())
    problems = []
    for g in groups:
        grp = store[g]
        if set(grp.keys()) != EXPECTED_SETS:
            problems.append((g, 'datasets', sorted(set(grp.keys()) ^ EXPECTED_SETS)))
        if grp['frames'].shape != (EXPECTED_FRAMES, TARGET_SIZE, TARGET_SIZE, 3):
            problems.append((g, 'frames shape', grp['frames'].shape))
        if grp['frames'].chunks != (FRAMES_PER_CHUNK, TARGET_SIZE, TARGET_SIZE, 3):
            problems.append((g, 'frames chunks', grp['frames'].chunks))
        if len(grp['frame_t_ms']) != EXPECTED_FRAMES or len(grp['frame_ecg_idx']) != EXPECTED_FRAMES:
            problems.append((g, 'frame-aligned length'))
        if any(len(grp[k]) != N_WINDOWS for k in REF_KEYS):
            problems.append((g, 'reference length'))
        if not np.array_equal(grp['ref_window_start'][:], np.arange(0, N_WINDOWS) * WINDOW_STRIDE):
            problems.append((g, 'window grid'))
        if grp.attrs['role'] != ROLE or 'bvp' in grp or 'cardiac_sqi' in grp.attrs:
            problems.append((g, 'guard broken'))
    qc = pd.DataFrame([dict(store[g].attrs) | {'group': g} for g in groups])

    # NB_P3_07's index filter, verbatim in effect: usable, cardiac_sqi >= 0.05 (missing
    # reads as 0.0), n_frames >= CLIP_LEN. Every group must fail it.
    admitted = [g for g in groups
                if bool(store[g].attrs.get('usable', True))
                and float(store[g].attrs.get('cardiac_sqi', 0.0)) >= 0.05
                and int(store[g].attrs['n_frames']) >= CLIP_LEN]

print(f'groups: {len(groups)} | schema problems: {problems or "none"}')
print(f'training-loader filter admits: {len(admitted)} of {len(groups)} '
      f'({"guard holds" if not admitted else "GUARD BROKEN"})')
print(f'store on disk: {OUT_H5.stat().st_size / 1e9:.2f} GB')
print()
print(f'usable (exposure): {int(qc.usable.sum())}/{len(qc)} | brightness '
      f'{qc.brightness.min():.0f}-{qc.brightness.max():.0f}')
print(f'face found in all {N_DETECT_FRAMES} samples: {int((qc.face_detections == N_DETECT_FRAMES).sum())}'
      f'/{len(qc)} | fewest: {int(qc.face_detections.min())}')
low_detection = qc.loc[qc.face_detections < N_DETECT_FRAMES / 2, 'group'].tolist()
print(f'face in fewer than half the samples (reported separately under the protocol): '
      f'{low_detection or "none"}')
print(f'fps from the capture clock: {qc.fps.min():.3f}-{qc.fps.max():.3f} | container declares '
      f'{sorted(qc.fps_container.round(3).unique().tolist())}')
print(f'windows with a real span more than {SPAN_TOLERANCE:.0%} off nominal: '
      f'{int(qc.ref_windows_off_span.sum())} in '
      f'{int((qc.ref_windows_off_span > 0).sum())} recording(s)')
print()
print('app capture verdict:', qc.capture_verdict.value_counts().to_dict())
clamped = qc[qc.app_frac_side_lost > MAX_BOX_SIDE_LOST]
print(f'crop clamped beyond {MAX_BOX_SIDE_LOST:.0%} (reported separately under the protocol):')
print(clamped[['group', 'app_frac_side_lost', 'box_side_lost', 'capture_verdict']]
      .round(3).to_string(index=False) if len(clamped) else '  none')
print()
accepted = int(qc.ref_accepted.sum())
recordings_ok = int(qc.ref_ok.sum())
print(f'referenced windows: {accepted} of {int(qc.ref_windows.sum())} | recordings at '
      f'{REF_RECORDING_MIN:.0%} or more: {recordings_ok} of {len(qc)}')
if len(qc) == 100:
    same = accepted == NB_P3_28_ACCEPTED and recordings_ok == NB_P3_28_RECORDINGS_OK
    print(f'against NB_P3_28 ({NB_P3_28_ACCEPTED}, {NB_P3_28_RECORDINGS_OK}): '
          f'{"reproduced exactly" if same else "DIFFERS - explain before benchmarking"}')
else:
    print(f'reproduction check needs all 100 recordings; {len(qc)} present')
print()
print('by activity:')
print(qc.groupby(['activity', 'lighting', 'motion_class'])
      .agg(n=('group', 'size'), ref_accepted_frac=('ref_accepted_frac', 'median'),
           ref_ok=('ref_ok', 'sum'), ref_hr_median=('ref_hr_median', 'median'),
           brightness=('brightness', 'median'))
      .round(3).to_string())

groups: 100 | schema problems: none
training-loader filter admits: 0 of 100 (guard holds)
store on disk: 2.86 GB

usable (exposure): 100/100 | brightness 19-163
face found in all 12 samples: 52/100 | fewest: 3
face in fewer than half the samples (reported separately under the protocol): ['00_02', '00_04', '01_06', '16_04']
fps from the capture clock: 28.158-30.016 | container declares [30.0]
windows with a real span more than 5% off nominal: 21 in 1 recording(s)

app capture verdict: {'ACCEPT': 96, 'REJECT': 4}
crop clamped beyond 2% (reported separately under the protocol):
group  app_frac_side_lost  box_side_lost capture_verdict
07_05               0.050          0.088          REJECT
08_05               0.052          0.093          REJECT
08_06               0.038          0.067          REJECT
10_06               0.036          0.064          REJECT

referenced windows: 1883 of 2100 | recordings at 80% or more: 83 of 100
against NB_P3_28 (1883, 83): reproduced exactly

by activity

## 6. Findings (2026-10-04)

**The store.** All 100 recordings written, no errors: `phase 3 benchmarks\ecg_fitness_phase3_rc_none.h5`, 2.86 GB, 82 min of build time at a median 55 s per recording (the final run added the last 22; an earlier run had written the first 78, and resumability skipped them). Every group passes the schema check: the expected datasets, 1800 frames of 72x72 in (32, 72, 72, 3) chunks, the frame-aligned arrays at 1800, and 21 reference windows on the 80-frame grid.

**The guards.** NB_P3_07's training filter admits 0 of 100: no group carries `bvp` or `cardiac_sqi`, every group carries `role='benchmark'`, and the store sits in `phase 3 benchmarks\`, outside every training glob.

**Reproduction.** 1883 of 2100 windows carry a reference and 83 of 100 recordings reach 80% - NB_P3_28's figures, reproduced exactly by the code that wrote the store.

**The container.** Every video declares 30 fps and decodes to exactly 1800 frames, one per sync row. The capture clock measures 28.158-30.016 fps, and two videos run more than 2% below the declared rate (01/01 at 28.76, 08/05 at 28.16), which is why the store's `fps` comes from the clock. Frame timing is regular almost everywhere - a median 10.8 ms rms from a uniform grid - with three exceptions: 08/05 (2226 ms), 01/01 (1225 ms) and 15/03 (297 ms). In 08/05 all 21 windows span more than 5% off nominal, the set Part 0 reports separately, and the recording is also below the reference gate. 01/01 and 15/03 vary slowly enough that every window stays within 5%.

**The crop on c920-2.** The face was found in all 12 sampled frames in 52 recordings and in at least 3 in every one. Four fall below half the samples - 00/02, 00/04 and 16/04, all rowing (02/04, on NB_P3_28's reading), and 01/06 - and are reported separately; the three rowing ones are also below the reference gate, so 01/06 is the only recording this rule takes out of the primary set. Four crops clamp beyond 2% of their side - 07/05, 08/05, 08/06 and 10/06, all on activities 05 and 06 - and the app's framing gate refuses all four; nothing else clamps at all, and the other 96 recordings pass the capture gates. The app's current `frac_side_lost` (0.036-0.052) and the per-axis `box_side_lost` (0.064-0.093) flag the same four, so the pending `frac_side_lost` fix changes no verdict on this corpus. Brightness runs 19-163, and the exposure check passes everywhere.

**The reference gate is not activity-neutral.** It passes 64 of the 68 recordings outside rowing but 19 of the 32 rowing ones (11 of 16 on 02, 8 of 16 on 04). With the separately reported recordings set aside, the primary set is 80 recordings - 16, 11, 17, 8, 14 and 14 on activities 01 to 06 - with 1596 of their 1680 windows referenced, and 18 of them running above 120 bpm. H3's rowing comparison, 04 against 02, therefore rests on 8 and 11 recordings, and activity 04 is the thinnest cell in the benchmark.

**Ready for the benchmark: yes.** Nothing is left to resolve: the primary set follows from the store alone, and every recording kept out of it was named by a rule in Part 0 before any estimator ran. The `frac_side_lost` fix does not need to come first, since it moves no recording across the 2% line here. The next step is `app/live_vitals/scripts/check_ecg_fitness.py`, implementing Part 0 as written, with both estimators run at the stored capture-clock `fps`.